# ClauseFinder — RAG sobre contratos legales

Notebook de la asignatura **Ingeniería de Soluciones con Inteligencia Artificial (012V)** — DUOC UC.

Responde preguntas sobre cláusulas contractuales **citando el texto literal** del contrato de origen.

**Pipeline:**

```
contratos → chunking por cláusula → índice denso + índice BM25
                                          ↓
pregunta → retrieval híbrido → fusión RRF → re-ranking → DeepSeek → respuesta + cita
                                                              ↓
                                                    verificación de la cita
```

**Cómo ejecutarlo:** de arriba hacia abajo. Los bloques 1 a 6 funcionan sin conexión y sin
credenciales. El bloque 7 (generación) necesita `DEEPSEEK_API_KEY` (se lee desde `.env`).

## 0. Dependencias

In [10]:
# Ejecutar una sola vez. Quitar el comentario si es la primera corrida.
# !pip install -q sentence-transformers rank_bm25 openai python-dotenv numpy pdfplumber

In [11]:
import os
import re
import json
import pathlib

import numpy as np

# --- Configuración -----------------------------------------------------------
DATA_RAW      = pathlib.Path("data/raw")      # contratos .txt o .pdf (opcional)
MODEL_EMB     = "intfloat/multilingual-e5-small"                # embeddings multilingües, CPU
MODEL_RERANK  = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"    # re-ranker multilingüe
MODEL_LLM     = "deepseek-chat"      # o "deepseek-reasoner" para razonamiento explícito
DEEPSEEK_URL  = "https://api.deepseek.com"

MAX_CHARS     = 4000   # techo por chunk (~1.000 tokens)
TOP_K_FUSION  = 30     # candidatos que salen de la fusión RRF
TOP_K_CONTEXT = 5      # fragmentos que entran al prompt
USE_RERANKER  = True

np.random.seed(42)
print("Configuración lista.")

Configuración lista.


## 1. Carga de documentos

El corpus son **cuatro contratos comerciales reales**, en PDF, tomados de
[CUAD](https://www.atticusprojectai.org/cuad) — que a su vez los extrae de filings públicos de la SEC
de Estados Unidos. Son documentos que se firmaron de verdad, con toda la irregularidad de formato que
eso implica.

| Archivo | Tipo | Partes | Año |
|---|---|---|---|
| `MSA_MeritLife_2020.pdf` | Master Services Agreement | Merit Life Insurance | 2020 |
| `CoBranding_Healthcentral_1999.pdf` | Co-Branding Agreement | Healthcentral.com / MLX | 1999 |
| `Licencia_Mantenimiento_SparklingSpring_2002.pdf` | Licencia de software y mantenimiento | Sparkling Spring / Garman | 2002 |
| `Mantenimiento_XImage_Sagem_1999.pdf` | Contrato de mantenimiento | XIMAGE / SAGEM Morpho | 1999 |

El pipeline no está atado a estos archivos: **cualquier PDF que se deje en `data/raw/` entra por el
mismo flujo**.

In [12]:
PDFS = sorted(DATA_RAW.glob("*.pdf")) + sorted(DATA_RAW.glob("*.txt"))

if not PDFS:
    print(f"No hay documentos en {DATA_RAW.resolve()}")
    print("Dejar ahí los PDF de contratos y volver a ejecutar esta celda.")
else:
    for ruta in PDFS:
        print(f"  {ruta.name:<52} {ruta.stat().st_size // 1024:>4} KB")

  CoBranding_Healthcentral_1999.pdf                      81 KB
  Licencia_Mantenimiento_SparklingSpring_2002.pdf        61 KB
  Mantenimiento_XImage_Sagem_1999.pdf                    58 KB
  MSA_MeritLife_2020.pdf                                142 KB


### 1.1 De PDF a texto utilizable

Extraer el texto es la parte fácil. Lo que rompe el pipeline es lo que viene pegado a él, y que en
estos contratos aparece todo junto:

- **Índice con puntos guía** (`3. Software Maintenance Services......4`). Cada línea del índice parece
  un encabezado de sección: sin filtrarlas, el contrato de XIMAGE generaba 16 fragmentos basura de 82
  caracteres y perdía su estructura real.
- **Encabezados y pies repetidos** en cada página (`Exhibit 10.22`, `Page 4 of 12`), que parten
  cláusulas por la mitad y ensucian los embeddings.

Ambos se detectan sin heurísticas frágiles: los puntos guía por su patrón, y los encabezados
repetidos porque son las líneas que aparecen en la mitad o más de las páginas.

El separador `\f` (salto de página) se conserva a propósito: es lo que después permite decir **en qué
página** está la cláusula citada.

In [13]:
import collections

RE_PUNTOS_GUIA = re.compile(r"\.{4,}\s*\d+\s*$")   # línea de índice: "Definitions.......3"


def extraer_texto(ruta):
    '''PDF -> texto, separando páginas con \f para poder citar el número de página.'''
    if ruta.suffix.lower() == ".txt":
        return ruta.read_text(encoding="utf-8", errors="ignore")

    import pdfplumber
    with pdfplumber.open(ruta) as pdf:
        return "\f".join((pagina.extract_text() or "") for pagina in pdf.pages)


def limpiar(texto):
    '''Quita el índice y los encabezados/pies que se repiten en casi todas las páginas.'''
    paginas = texto.split("\f")

    frecuencia = collections.Counter()
    for pagina in paginas:
        for linea in {l.strip() for l in pagina.split("\n") if 3 < len(l.strip()) < 80}:
            frecuencia[linea] += 1

    umbral = max(2, int(len(paginas) * 0.5))
    ruido  = {linea for linea, n in frecuencia.items() if n >= umbral}

    limpias = []
    for pagina in paginas:
        limpias.append("\n".join(
            l for l in pagina.split("\n")
            if not RE_PUNTOS_GUIA.search(l) and l.strip() not in ruido
        ))
    return "\f".join(limpias)

In [14]:
def cargar_documentos():
    docs = []
    for ruta in PDFS:
        texto = limpiar(extraer_texto(ruta))
        docs.append({
            "doc_id":  ruta.stem,
            "texto":   texto,
            "titulo":  next((l.strip() for l in texto.split("\n") if l.strip()), ruta.stem)[:90],
            "paginas": texto.count("\f") + 1,
        })
    return docs


documentos = cargar_documentos()
for d in documentos:
    print(f"  {d['doc_id']:<46} {d['paginas']:>2} pág  {len(d['texto']):>6} car.")
    print(f"     └─ {d['titulo'][:80]}")

  CoBranding_Healthcentral_1999                  13 pág   40302 car.
     └─ EXHIBIT 10.27
  Licencia_Mantenimiento_SparklingSpring_2002     7 pág   24796 car.
     └─ Exhibit 10.13
  Mantenimiento_XImage_Sagem_1999                 6 pág   18208 car.
     └─ MAINTENANCE AGREEMENT
  MSA_MeritLife_2020                             10 pág   22035 car.
     └─ Exhibit 10(xiv)


## 2. Chunking por cláusula

**Esta es la decisión de diseño que más pesa en la calidad final.**

El *chunking* de tamaño fijo (512 tokens con solape) es el default de la mayoría de los tutoriales y
**es la peor opción para contratos**: parte cláusulas por la mitad y mezcla obligaciones de secciones
distintas en un mismo fragmento. En contratos eso produce respuestas que suenan bien y son falsas.

Aquí se corta en los **límites estructurales reales**: los encabezados de artículo.

**Granularidad: artículo, no subcláusula.** Cortar en cada `3.1`, `3.2` da fragmentos más precisos
pero rompe el sentido: una cláusula como *"The indemnification obligations in Section 5.1 shall not
apply..."* no significa nada sin la 5.1. Manteniendo el artículo completo, la regla y su excepción
viajan juntas — que es justamente el error más común en RAG legal. Solo si un artículo supera el techo
de tokens se parte, y se parte por subincisos `(a)`, `(b)`, nunca por conteo de caracteres.

### El problema real: hay varias numeraciones compitiendo

En un contrato de verdad, `1.`, `2.`, `3.` aparece en varios lugares a la vez: los recitales
(*"Whereas: 1... 2..."*), las listas dentro de una cláusula y los artículos. Un detector ingenuo los
mezcla todos y la segmentación queda inservible.

La solución es simple y robusta: la numeración verdadera es **la que cubre el documento entero**. Se
prueba cada arranque posible en `1` y gana la secuencia ascendente que más secciones encadena — los
recitales dan una corrida de dos o tres, los artículos dan una de veinte. Se tolera además un salto de
un número, para absorber los encabezados que la extracción de PDF se come.

In [15]:
# "3. TERMINATION", "ARTICLE V", "SECTION 8 —"
RE_ENCABEZADO = re.compile(
    r"^\s*((?:ARTICLE|SECTION|ART[IÍ]CULO|SECCI[OÓ]N)\s+[IVXLC0-9]+|[0-9]+(?:\.[0-9]+)*)"
    r"\s*[.\-—:)]?\s+(.*)$"
)
# el PDF a veces deja el número solo en una línea y el título en la siguiente
RE_SOLO_NUMERO = re.compile(r"^\s*([0-9]{1,2}|(?:ARTICLE|SECTION)\s+[IVXLC0-9]+)\s*\.?\s*$", re.I)
# subincisos, solo para partir artículos demasiado largos
RE_SUBINCISO = re.compile(r"(?=\((?:[a-z]|[ivx]+|[0-9])\)\s)")

ROMANOS = {r: i + 1 for i, r in enumerate(
    "I II III IV V VI VII VIII IX X XI XII XIII XIV XV XVI XVII XVIII XIX XX "
    "XXI XXII XXIII XXIV XXV".split())}


def _valor(numero):
    '''Valor numérico del encabezado: "7" -> 7, "ARTICLE V" -> 5.'''
    partes = numero.upper().split()
    if len(partes) == 2:
        return ROMANOS.get(partes[1], int(partes[1]) if partes[1].isdigit() else None)
    return int(numero) if numero.isdigit() else None


def candidatos(texto):
    lineas, pos, encontrados = texto.split("\n"), 0, []
    for i, linea in enumerate(lineas):
        numero = titulo = None
        m = RE_ENCABEZADO.match(linea)
        if m and len(linea.strip()) > 3:
            numero, titulo = m.group(1).strip(), m.group(2).strip()
        else:
            m2 = RE_SOLO_NUMERO.match(linea)
            if m2 and i + 1 < len(lineas) and lineas[i + 1].strip():
                numero, titulo = m2.group(1).strip(), lineas[i + 1].strip()
        if numero:
            encontrados.append({"inicio": pos, "numero": numero, "titulo": titulo,
                                "valor": _valor(numero)})
        pos += len(linea) + 1
    return encontrados


def mejor_secuencia(cands):
    '''La numeración de artículos es la corrida ascendente más larga que arranca en 1.'''
    nivel1 = [c for c in cands if c["valor"] is not None and "." not in c["numero"]]
    mejor = []
    for i, arranque in enumerate(nivel1):
        if arranque["valor"] != 1:
            continue
        corrida, esperado = [arranque], 2
        for c in nivel1[i + 1:]:
            if c["valor"] == esperado:
                corrida.append(c); esperado += 1
            elif c["valor"] == esperado + 1:      # encabezado perdido en la extracción
                corrida.append(c); esperado = c["valor"] + 1
        if len(corrida) > len(mejor):
            mejor = corrida
    return mejor

In [16]:
def segmentar(doc):
    texto  = doc["texto"]
    cands  = candidatos(texto)
    cortes = mejor_secuencia(cands)
    if len(cortes) < 3:   # contrato sin numeración reconocible
        cortes = [c for c in cands if c["valor"] is not None] \
                 or [{"inicio": 0, "numero": "—", "titulo": doc["titulo"]}]

    secciones = []
    for i, c in enumerate(cortes):
        fin = cortes[i + 1]["inicio"] if i + 1 < len(cortes) else len(texto)
        seccion = {**c, "fin": fin, "texto": texto[c["inicio"]:fin].strip()}
        if seccion["texto"]:
            secciones.append(seccion)
    return secciones


def trocear_largos(seccion):
    '''Si un artículo supera el techo se parte por subincisos, nunca por caracteres.'''
    if len(seccion["texto"]) <= MAX_CHARS:
        return [seccion]

    partes, acumulado, inicio = [], "", seccion["inicio"]
    for trozo in RE_SUBINCISO.split(seccion["texto"]):
        if len(acumulado) + len(trozo) > MAX_CHARS and acumulado:
            partes.append({**seccion, "texto": acumulado.strip(),
                           "inicio": inicio, "fin": inicio + len(acumulado)})
            inicio += len(acumulado)
            acumulado = trozo
        else:
            acumulado += trozo
    if acumulado.strip():
        partes.append({**seccion, "texto": acumulado.strip(),
                       "inicio": inicio, "fin": inicio + len(acumulado)})
    return partes


def construir_chunks(documentos):
    chunks = []
    for doc in documentos:
        for seccion in segmentar(doc):
            for parte in trocear_largos(seccion):
                ruta = f"{doc['titulo']} > {parte['numero']} {parte['titulo']}"
                chunks.append({
                    "chunk_id": f"{doc['doc_id']}::{parte['numero']}::{len(chunks)}",
                    "doc_id":   doc["doc_id"],
                    "titulo":   doc["titulo"],
                    "seccion":  parte["numero"],
                    "epigrafe": parte["titulo"][:70],
                    "pagina":   doc["texto"].count("\f", 0, parte["inicio"]) + 1,
                    "inicio":   parte["inicio"],
                    "fin":      parte["fin"],
                    "texto":    parte["texto"],
                    # se indexa con la ruta jerárquica: la cláusula sola es ininteligible
                    "texto_indexado": f"{ruta}\n{parte['texto']}",
                })
    return chunks


chunks = construir_chunks(documentos)
print(f"{len(chunks)} chunks a partir de {len(documentos)} contratos")
print(f"Largo promedio: {np.mean([len(c['texto']) for c in chunks]):.0f} caracteres")
print(f"Largo máximo:   {max(len(c['texto']) for c in chunks)} caracteres")

88 chunks a partir de 4 contratos
Largo promedio: 1157 caracteres
Largo máximo:   3957 caracteres


In [17]:
# Estructura detectada en cada contrato
for doc in documentos:
    propios = [c for c in chunks if c["doc_id"] == doc["doc_id"]]
    print(f"\n{doc['doc_id']}  ({len(propios)} secciones)")
    for c in propios:
        print(f"   [{c['seccion']:>3}] p{c['pagina']:<2} {len(c['texto']):>5}c  {c['epigrafe'][:58]}")


CoBranding_Healthcentral_1999  (30 secciones)
   [  1] p1    529c  SERVICES & DEVELOPMENT
   [  2] p1   3030c  MLX'S OBLIGATIONS AND RESPONSIBILITIESDuring the Term of 
   [  3] p2   3441c  HCI's OBLIGATIONS AND RESPONSIBILITIES
   [  3] p3    888c  HCI's OBLIGATIONS AND RESPONSIBILITIES
   [  4] p4   1453c  ADVERTISING
   [  5] p4    757c  E-COMMERCE
   [  6] p4   2343c  FINANCIALS
   [  7] p5   1238c  CUSTOMER INFORMATION
   [  8] p5    587c  PAGE VIEWS
   [  9] p5   1643c  EXCLUSIVITY
   [ 10] p6    888c  BRANDING & COMMUNICATIONS
   [ 11] p6   2261c  TERM & TERMINATION
   [ 12] p7   3957c  REPRESENTATIONS & WARRANTIES
   [ 12] p8   1401c  REPRESENTATIONS & WARRANTIES
   [ 13] p8   1443c  NO ADDITIONAL WARRANTIES
   [ 14] p8   1888c  INDEMNIFICATION(a) By MLX: MLX agrees to indemnify and ho
   [ 15] p9    735c  LIMITATION OF LIABILITY
   [ 16] p9   1089c  CONFIDENTIALITY
   [ 17] p9    410c  NO PUBLIC NOTICENo public announcement of the proposed tr
   [ 18] p10   257c  COSTS
   

## 3. Índice denso (embeddings)

**El modelo tiene que ser multilingüe.** El corpus está en inglés y las preguntas se hacen en
español: con `bge-small-en-v1.5` (solo inglés) el retrieval acierta 2 de 4 preguntas de prueba; con
`multilingual-e5-small` acierta 4 de 4, todas en primer lugar. Es la diferencia entre un sistema que
funciona y uno que no, y no se nota hasta que se mide.

Los vectores se guardan en una matriz NumPy normalizada: con este volumen la búsqueda exacta por
producto punto es instantánea y evita montar un servidor. Para decenas de miles de contratos se
cambia por FAISS o ChromaDB sin tocar el resto del pipeline.

> La API de DeepSeek **no expone un endpoint de embeddings**: el retrieval se resuelve con un modelo
> local (como aquí) o con un proveedor externo. Para más calidad: `BAAI/bge-m3` (multilingüe, 2,2 GB)
> o `voyage-law-2` (entrenado en dominio legal, de pago).

In [18]:
from sentence_transformers import SentenceTransformer

modelo_emb = SentenceTransformer(MODEL_EMB)

# La familia E5 exige estos prefijos: la consulta y el documento se codifican distinto.
PREFIJO_QUERY = "query: "
PREFIJO_DOC   = "passage: "

E = modelo_emb.encode(
    [PREFIJO_DOC + c["texto_indexado"] for c in chunks],
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(f"\nMatriz de embeddings: {E.shape}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1520.70it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3/3 [00:16<00:00,  5.60s/it]


Matriz de embeddings: (88, 384)


## 4. Índice léxico (BM25)

Los contratos están llenos de términos exactos que la búsqueda semántica pierde: `Force Majeure`,
`indemnitee`, `thirty (30) days`. BM25 los captura literalmente.

**Advertencia honesta:** BM25 es monolingüe por definición. Con preguntas en español sobre contratos
en inglés aporta poco —solo rescata los términos que el usuario escribe en inglés, que en jerga legal
son muchos ("¿aplica el *force majeure*?")—. Sobre un corpus en español aporta plenamente. En la
evaluación del bloque 8 se ve exactamente cuánto suma en este escenario.

In [19]:
from rank_bm25 import BM25Okapi


def tokenizar(texto):
    return re.findall(r"[a-z0-9]+", texto.lower())


bm25 = BM25Okapi([tokenizar(c["texto_indexado"]) for c in chunks])
print(f"Índice BM25 construido sobre {len(chunks)} chunks.")

Índice BM25 construido sobre 88 chunks.


## 5. Retrieval híbrido + fusión RRF

Ninguna de las dos búsquedas gana sola: la densa entiende paráfrasis ("¿puedo salirme del
contrato?" → cláusula de terminación), la léxica encuentra el término exacto. **Reciprocal Rank
Fusion** las combina sin necesidad de normalizar puntajes de escalas distintas.

$$\text{RRF}(d) = \sum_{r \in \text{rankings}} \frac{1}{k + \text{rank}_r(d)}, \quad k = 60$$

In [20]:
def buscar_denso(pregunta, k=TOP_K_FUSION):
    q = modelo_emb.encode([PREFIJO_QUERY + pregunta], normalize_embeddings=True)[0]
    puntajes = E @ q                       # coseno, porque los vectores están normalizados
    return list(np.argsort(-puntajes)[:k])


def buscar_bm25(pregunta, k=TOP_K_FUSION):
    puntajes = bm25.get_scores(tokenizar(pregunta))
    return list(np.argsort(-puntajes)[:k])


def fusion_rrf(rankings, k=60):
    acumulado = {}
    for ranking in rankings:
        for posicion, idx in enumerate(ranking):
            acumulado[idx] = acumulado.get(idx, 0.0) + 1.0 / (k + posicion + 1)
    return sorted(acumulado, key=acumulado.get, reverse=True)


def recuperar(pregunta, k=TOP_K_FUSION):
    return fusion_rrf([buscar_denso(pregunta, k), buscar_bm25(pregunta, k)])[:k]


pregunta_demo = "En el contrato de XIMAGE con SAGEM, ¿en cuánto tiempo debe responder XIMAGE?"
for idx in recuperar(pregunta_demo)[:5]:
    c = chunks[idx]
    print(f"{c['doc_id']:<24} [{c['seccion']}] {c['epigrafe'][:40]}")

Mantenimiento_XImage_Sagem_1999 [8] RESPONSE BY XIMAGE.
Mantenimiento_XImage_Sagem_1999 [20] ENTIRE AGREEMENT AND WAIVERThis Agreeme
Mantenimiento_XImage_Sagem_1999 [11] INJUNCTIVE RELIEF/TERMINATION
Mantenimiento_XImage_Sagem_1999 [9] INTERVENTION ON SITE.In the event that 
Mantenimiento_XImage_Sagem_1999 [10] RETURN AND REPAIR


## 6. Re-ranking

El retrieval híbrido es rápido pero impreciso en el orden. Un *cross-encoder* lee cada par
(pregunta, fragmento) junto y reordena los 30 candidatos para quedarse con los 5 mejores. Es más
lento, por eso se aplica solo al final.

In [21]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(MODEL_RERANK) if USE_RERANKER else None


def reordenar(pregunta, indices, k=TOP_K_CONTEXT):
    if reranker is None:
        return indices[:k]
    pares = [(pregunta, chunks[i]["texto_indexado"]) for i in indices]
    puntajes = reranker.predict(pares)
    orden = np.argsort(-puntajes)
    return [indices[i] for i in orden[:k]]


def pipeline_retrieval(pregunta):
    return reordenar(pregunta, recuperar(pregunta))


print("Top-5 tras re-ranking:\n")
for pos, idx in enumerate(pipeline_retrieval(pregunta_demo), 1):
    c = chunks[idx]
    print(f"{pos}. {c['doc_id']:<24} [{c['seccion']}] {c['epigrafe'][:40]}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1400.15it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Top-5 tras re-ranking:

1. Mantenimiento_XImage_Sagem_1999 [8] RESPONSE BY XIMAGE.
2. Mantenimiento_XImage_Sagem_1999 [2] TERM OF AGREEMENT
3. Mantenimiento_XImage_Sagem_1999 [15] LIMITATIONS OF LIABILITY
4. Mantenimiento_XImage_Sagem_1999 [9] INTERVENTION ON SITE.In the event that 
5. CoBranding_Healthcentral_1999 [3] HCI's OBLIGATIONS AND RESPONSIBILITIES


## 7. Generación con DeepSeek

El prompt de sistema exige tres cosas: responder **solo** con los fragmentos recuperados, incluir la
**cita literal** con documento y sección, y decir explícitamente *"No encontrado"* cuando el contexto
no alcanza. La abstención es una respuesta correcta, no un fallo.

DeepSeek expone una API compatible con OpenAI, así que se usa el SDK `openai` apuntando a
`https://api.deepseek.com`. Se usa `deepseek-chat` (DeepSeek-V3.2, modo sin razonamiento) por costo y
latencia; para preguntas con obligaciones, excepciones y condiciones anidadas se puede cambiar
`MODEL_LLM` a `deepseek-reasoner` sin tocar el resto del código.

In [22]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # carga DEEPSEEK_API_KEY desde .env
client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url=DEEPSEEK_URL)

SYSTEM_PROMPT = '''Eres un asistente de análisis contractual. Respondes preguntas sobre cláusulas
usando EXCLUSIVAMENTE los fragmentos de contrato que se te entregan.

Reglas obligatorias:
1. No uses conocimiento externo. Si la respuesta no está en los fragmentos, responde exactamente:
   "No encontrado en los documentos proporcionados."
2. Toda afirmación debe ir acompañada de una cita LITERAL entre comillas, copiada carácter por
   carácter del fragmento, seguida de su origen en el formato (documento, sección).
3. Si dos contratos regulan el punto de forma distinta, indícalo explícitamente en lugar de
   promediar o generalizar.
4. Responde en español, de forma breve y directa. No agregues advertencias legales genéricas.
5. No eres asesor legal: describes lo que dice el contrato, no recomiendas cursos de acción.'''


def armar_contexto(indices):
    bloques = []
    for n, idx in enumerate(indices, 1):
        c = chunks[idx]
        bloques.append(
            f"[F{n}] documento: {c['doc_id']} | sección: {c['seccion']} {c['epigrafe']} | "
            f"página: {c['pagina']}\n{c['texto']}"
        )
    return "\n\n---\n\n".join(bloques)

In [23]:
def generar(pregunta, contexto):
    mensaje = f"""FRAGMENTOS RECUPERADOS:

{contexto}

PREGUNTA: {pregunta}"""

    respuesta = client.chat.completions.create(
        model=MODEL_LLM,
        max_tokens=8000,
        temperature=0.0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": mensaje},
        ],
    )

    eleccion = respuesta.choices[0]

    if eleccion.finish_reason == "content_filter":
        return "[El modelo declinó responder a esta consulta.]", respuesta.usage

    return eleccion.message.content or "", respuesta.usage

### 7.1 Verificación de citas

RAG **reduce** la alucinación, no la elimina. Este validador es la última defensa: comprueba que cada
texto entrecomillado exista carácter por carácter en algún fragmento recuperado. Es barato y detecta
justo el error más peligroso —la cláusula inventada que suena plausible.

In [24]:
RE_CITA = re.compile(r'"([^"]{15,})"')


def normalizar(t):
    return re.sub(r"\s+", " ", t).strip().lower()


def verificar_citas(respuesta, indices):
    fuente = normalizar(" ".join(chunks[i]["texto"] for i in indices))
    resultados = [(cita, normalizar(cita) in fuente) for cita in RE_CITA.findall(respuesta)]
    return resultados, all(ok for _, ok in resultados)

In [25]:
def preguntar(pregunta, verbose=True):
    indices  = pipeline_retrieval(pregunta)
    contexto = armar_contexto(indices)
    texto, uso = generar(pregunta, contexto)
    citas, todas_ok = verificar_citas(texto, indices)

    if verbose:
        print(f"PREGUNTA: {pregunta}\n")
        print(texto)
        print("\n" + "─" * 78)
        print("Fuentes consultadas: " + ", ".join(
            f"{chunks[i]['doc_id']} §{chunks[i]['seccion']}" for i in indices))
        if citas:
            estado = "todas las citas verificadas" if todas_ok else "HAY CITAS NO VERIFICADAS"
            print(f"Verificación: {len(citas)} cita(s) — {estado}")
            for cita, ok in citas:
                if not ok:
                    print(f"   ⚠ no encontrada en la fuente: \"{cita[:70]}...\"")
        else:
            print("Verificación: sin citas literales (esperable si respondió 'No encontrado')")
        print(f"Tokens: {uso.prompt_tokens} entrada / {uso.completion_tokens} salida")

    return {"respuesta": texto, "indices": indices, "citas": citas, "verificado": todas_ok}

In [26]:
# Requiere DEEPSEEK_API_KEY configurada en .env
_ = preguntar("En el acuerdo de co-branding de Healthcentral, ¿hay cláusula de exclusividad?")

PREGUNTA: En el acuerdo de co-branding de Healthcentral, ¿hay cláusula de exclusividad?

Sí. El contrato contiene una cláusula de exclusividad.

"9. EXCLUSIVITY
During the Term of the Agreement, and subject to the following requirements respecting Canadian content levels as set out in paragraphs 3(f), (g) and (h), HCI shall be the exclusive health content partner in the health section of the Sympatico web site" (CoBranding_Healthcentral_1999, sección 9 EXCLUSIVITY).

La exclusividad está condicionada y limitada:

- "provided however, that nothing contained herein shall prevent MediaLinx from entering into an agreement with other parties for: i) a health based web directory; or ii) other health related content whose content does not compete with the content contained on the Co-Branded Site; which in no event shall receive no more than one fifth (1/5th) of the promotion and which shall constitute no more than one fifth (1/5th) of the total health related content which resides on the home

In [27]:
_ = preguntar("En la licencia de Sparkling Spring, ¿existe depósito del código fuente en escrow?")

PREGUNTA: En la licencia de Sparkling Spring, ¿existe depósito del código fuente en escrow?

Sí. El contrato regula un depósito de código fuente, aunque no lo denomina "escrow" sino que establece que Garman debe mantener el código en cada AS400.

- "Immediately upon execution of this Agreement, Garman shall maintain on each AS400 at an Authorized Location the most current version of: (1) all source code for the Software in machine readable form, with user option to print" (Licencia_Mantenimiento_SparklingSpring_2002, sección 16 SOURCE CODE ESCROW).
- "This Source Code shall remain the property of Garman unless Sparkling exercises its option to purchase as specified in the Source Code Agreement." (Licencia_Mantenimiento_SparklingSpring_2002, sección 16 SOURCE CODE ESCROW).
- "Garman hereby grants to Sparkling, a perpetual, fully paid, non-exclusive license entitling Sparkling to use and reproduce the Codes deposited with it pursuant to Section (a)" (Licencia_Mantenimiento_SparklingSprin

In [28]:
# Caso de control: la respuesta NO está en el corpus. Debe abstenerse.
_ = preguntar("¿Cuál es el salario mínimo que debe pagar el proveedor a sus trabajadores?")

PREGUNTA: ¿Cuál es el salario mínimo que debe pagar el proveedor a sus trabajadores?

No encontrado en los documentos proporcionados.

──────────────────────────────────────────────────────────────────────────────
Fuentes consultadas: MSA_MeritLife_2020 §7, MSA_MeritLife_2020 §14, Licencia_Mantenimiento_SparklingSpring_2002 §10, MSA_MeritLife_2020 §5, MSA_MeritLife_2020 §1
Verificación: sin citas literales (esperable si respondió 'No encontrado')
Tokens: 2841 entrada / 9 salida


## 8. Evaluación

Se miden **retrieval y generación por separado**, porque un fallo en cada etapa se corrige distinto.

Para el retrieval se comparan cuatro estrategias contra un set de preguntas con la sección correcta
anotada. Sin estas líneas base no hay forma de defender que la arquitectura aporta algo: el punto es
mostrar *cuánto* mejora cada componente.

In [29]:
# (pregunta en español, doc_id esperado, artículo esperado)
#
# Las preguntas nombran el contrato porque la métrica exige acertar documento Y artículo:
# "¿hay cláusula de confidencialidad?" es correcta en los cuatro contratos a la vez.
# Es además el uso real: el analista sabe qué contrato está revisando.
EVAL = [
    # Master Services Agreement — Merit Life (2020)
    ("En el MSA de Merit Life, ¿cómo se compensa al contratista?",                        "MSA_MeritLife_2020", "3"),
    ("En el MSA de Merit Life, ¿quién es dueño de los materiales producidos?",            "MSA_MeritLife_2020", "4"),
    ("En el MSA de Merit Life, ¿qué se considera información confidencial?",              "MSA_MeritLife_2020", "8"),
    ("En el MSA de Merit Life, ¿se puede ceder o subcontratar el contrato?",              "MSA_MeritLife_2020", "10"),
    ("En el MSA de Merit Life, ¿qué seguros debe mantener el contratista?",               "MSA_MeritLife_2020", "14"),
    ("En el MSA de Merit Life, ¿cómo se resuelven las disputas?",                         "MSA_MeritLife_2020", "19"),

    # Co-Branding Agreement — Healthcentral.com (1999)
    ("En el acuerdo de co-branding de Healthcentral, ¿cómo se manejan los ingresos por publicidad?", "CoBranding_Healthcentral_1999", "4"),
    ("En el acuerdo de co-branding de Healthcentral, ¿qué pasa con la información de los clientes?", "CoBranding_Healthcentral_1999", "7"),
    ("En el acuerdo de co-branding de Healthcentral, ¿hay cláusula de exclusividad?",                "CoBranding_Healthcentral_1999", "9"),
    ("En el acuerdo de co-branding de Healthcentral, ¿cuál es el plazo y cómo se termina?",          "CoBranding_Healthcentral_1999", "11"),
    ("En el acuerdo de co-branding de Healthcentral, ¿quién indemniza frente a reclamos de terceros?", "CoBranding_Healthcentral_1999", "14"),
    ("En el acuerdo de co-branding de Healthcentral, ¿cuál es el límite de responsabilidad?",        "CoBranding_Healthcentral_1999", "15"),
    ("En el acuerdo de co-branding de Healthcentral, ¿qué ley lo rige?",                             "CoBranding_Healthcentral_1999", "21"),

    # Licencia de software y mantenimiento — Sparkling Spring (2002)
    ("En la licencia de Sparkling Spring, ¿qué restricciones de uso tiene el software?",             "Licencia_Mantenimiento_SparklingSpring_2002", "3"),
    ("En la licencia de Sparkling Spring, ¿se puede rechazar el software tras las pruebas?",         "Licencia_Mantenimiento_SparklingSpring_2002", "7"),
    ("En la licencia de Sparkling Spring, ¿qué cubren los servicios de mantenimiento y soporte?",    "Licencia_Mantenimiento_SparklingSpring_2002", "10"),
    ("En la licencia de Sparkling Spring, ¿qué obligaciones de confidencialidad hay?",               "Licencia_Mantenimiento_SparklingSpring_2002", "14"),
    ("En la licencia de Sparkling Spring, ¿existe depósito del código fuente en escrow?",            "Licencia_Mantenimiento_SparklingSpring_2002", "16"),
    ("En la licencia de Sparkling Spring, ¿qué secciones sobreviven a la terminación?",              "Licencia_Mantenimiento_SparklingSpring_2002", "18"),
    ("En la licencia de Sparkling Spring, ¿se puede contratar a empleados de la otra parte?",        "Licencia_Mantenimiento_SparklingSpring_2002", "19"),

    # Contrato de mantenimiento — XIMAGE / SAGEM Morpho (1999)
    ("En el contrato de XIMAGE con SAGEM, ¿hay obligación de no divulgación?",            "Mantenimiento_XImage_Sagem_1999", "7"),
    ("En el contrato de XIMAGE con SAGEM, ¿en cuánto tiempo debe responder XIMAGE?",      "Mantenimiento_XImage_Sagem_1999", "8"),
    ("En el contrato de XIMAGE con SAGEM, ¿qué pasa si se requiere intervención en terreno?", "Mantenimiento_XImage_Sagem_1999", "9"),
    ("En el contrato de XIMAGE con SAGEM, ¿qué garantía se otorga sobre el software?",    "Mantenimiento_XImage_Sagem_1999", "14"),
    ("En el contrato de XIMAGE con SAGEM, ¿cuáles son los límites de responsabilidad?",   "Mantenimiento_XImage_Sagem_1999", "15"),
    ("En el contrato de XIMAGE con SAGEM, ¿cómo se resuelven los conflictos?",            "Mantenimiento_XImage_Sagem_1999", "16"),
]
print(f"{len(EVAL)} preguntas de evaluación sobre {len(documentos)} contratos.")

26 preguntas de evaluación sobre 4 contratos.


In [30]:
def acierta(idx, doc_esperado, seccion_esperada):
    c = chunks[idx]
    return c["doc_id"] == doc_esperado and c["seccion"] == seccion_esperada


def evaluar(estrategia, k=10):
    recalls, reciprocos = [], []
    for pregunta, doc, seccion in EVAL:
        indices = estrategia(pregunta)[:k]
        posiciones = [i for i, idx in enumerate(indices) if acierta(idx, doc, seccion)]
        recalls.append(1.0 if posiciones else 0.0)
        reciprocos.append(1.0 / (posiciones[0] + 1) if posiciones else 0.0)
    return np.mean(recalls), np.mean(reciprocos)


ESTRATEGIAS = {
    "Solo BM25 (léxico)":      lambda q: buscar_bm25(q),
    "Solo denso (embeddings)": lambda q: buscar_denso(q),
    "Híbrido (RRF)":           lambda q: recuperar(q),
    "Híbrido + re-ranking":    lambda q: reordenar(q, recuperar(q), k=10),
}

print(f"{'Estrategia':<26}{'Recall@10':>12}{'MRR':>10}")
print("─" * 48)
resultados = {}
for nombre, estrategia in ESTRATEGIAS.items():
    recall, mrr = evaluar(estrategia)
    resultados[nombre] = (recall, mrr)
    print(f"{nombre:<26}{recall:>12.3f}{mrr:>10.3f}")

Estrategia                   Recall@10       MRR
────────────────────────────────────────────────
Solo BM25 (léxico)               0.385     0.107
Solo denso (embeddings)          0.692     0.458
Híbrido (RRF)                    0.577     0.241
Híbrido + re-ranking             0.808     0.491


In [31]:
# Recall@k del pipeline completo. Se recupera UNA vez por pregunta y se evalúan todos los k
# sobre el mismo ranking: recalcular el re-ranking por cada k cuesta minutos y no cambia nada.
rankings = [reordenar(q, recuperar(q), k=10) for q, _, _ in EVAL]

print(f"{'k':>4}{'Recall@k':>12}")
print("─" * 16)
for k in (1, 3, 5, 10):
    aciertos = [
        any(acierta(idx, doc, sec) for idx in ranking[:k])
        for ranking, (_, doc, sec) in zip(rankings, EVAL)
    ]
    print(f"{k:>4}{np.mean(aciertos):>12.3f}")

   k    Recall@k
────────────────
   1       0.385
   3       0.500
   5       0.654
  10       0.808


In [32]:
# Dónde falla: preguntas cuyo artículo correcto no quedó en el top-5
print("Preguntas sin el artículo esperado en el top-5:\n")
for ranking, (pregunta, doc, sec) in zip(rankings, EVAL):
    if not any(acierta(idx, doc, sec) for idx in ranking[:5]):
        primero = chunks[ranking[0]]
        print(f"  {pregunta[:66]}")
        print(f"     esperado: {doc} §{sec}")
        print(f"     devuelto: {primero['doc_id']} §{primero['seccion']} {primero['epigrafe'][:40]}\n")

Preguntas sin el artículo esperado en el top-5:

  En el MSA de Merit Life, ¿se puede ceder o subcontratar el contrat
     esperado: MSA_MeritLife_2020 §10
     devuelto: CoBranding_Healthcentral_1999 §2 MLX'S OBLIGATIONS AND RESPONSIBILITIESD

  En el acuerdo de co-branding de Healthcentral, ¿cuál es el plazo y
     esperado: CoBranding_Healthcentral_1999 §11
     devuelto: CoBranding_Healthcentral_1999 §10 BRANDING & COMMUNICATIONS

  En el acuerdo de co-branding de Healthcentral, ¿quién indemniza fr
     esperado: CoBranding_Healthcentral_1999 §14
     devuelto: CoBranding_Healthcentral_1999 §3 HCI's OBLIGATIONS AND RESPONSIBILITIES

  En el acuerdo de co-branding de Healthcentral, ¿cuál es el límite 
     esperado: CoBranding_Healthcentral_1999 §15
     devuelto: CoBranding_Healthcentral_1999 §3 HCI's OBLIGATIONS AND RESPONSIBILITIES

  En el acuerdo de co-branding de Healthcentral, ¿qué ley lo rige?
     esperado: CoBranding_Healthcentral_1999 §21
     devuelto: CoBranding_Health

## 9. Costo

| Modelo | ID | Entrada $/1M (sin caché) | Entrada $/1M (caché) | Salida $/1M |
|---|---|---|---|---|
| DeepSeek-V3.2 | `deepseek-chat` | $0,28 | $0,028 | $0,42 |
| DeepSeek-V3.2 (razonamiento) | `deepseek-reasoner` | $0,28 | $0,028 | $0,42 |

Precios de lista según <https://api-docs.deepseek.com/quick_start/pricing>; conviene verificarlos
antes de presupuestar, cambian con frecuencia.

Una consulta típica de este notebook usa ~2.000 tokens de entrada y ~400 de salida: alrededor de
**US$ 0,0007** con `deepseek-chat`. Las 26 preguntas de evaluación cuestan menos de dos centavos.

Dos observaciones sobre el costo:

- **El caché de prefijo es automático** en DeepSeek: el `SYSTEM_PROMPT` se repite en cada consulta y
  se cobra a la tarifa de caché (10× más barata) cuando acierta, sin configurar nada.
- **`deepseek-reasoner`** genera tokens de razonamiento que también se cobran como salida; una
  consulta puede multiplicar por 5–10 los tokens de salida respecto a `deepseek-chat`.

In [33]:
def estimar_costo(tokens_entrada, tokens_salida, precio_in=0.28, precio_out=0.42):
    return tokens_entrada / 1e6 * precio_in + tokens_salida / 1e6 * precio_out


print(f"1 consulta      : US$ {estimar_costo(2000, 400):.4f}")
print(f"26 evaluaciones : US$ {estimar_costo(2000 * 26, 400 * 26):.4f}")
print(f"1.000 consultas : US$ {estimar_costo(2000 * 1000, 400 * 1000):.2f}")

1 consulta      : US$ 0.0007
26 evaluaciones : US$ 0.0189
1.000 consultas : US$ 0.73


## 10. Conclusiones y limitaciones

### Lo que muestra la evaluación

Estos son los números medidos sobre los cuatro contratos reales, con 26 preguntas en español:

| Estrategia | Recall@10 | MRR |
|---|---|---|
| Solo BM25 (léxico) | 0,385 | 0,107 |
| Solo denso (embeddings) | 0,692 | 0,458 |
| Híbrido (RRF) | 0,577 | 0,241 |
| **Híbrido + re-ranking** | **0,808** | **0,491** |

**El resultado incómodo: la fusión RRF por sí sola empeora las cosas** (0,577 contra 0,692 del índice
denso solo). No es un error de implementación, es consecuencia del escenario: las preguntas van en
español y los contratos están en inglés, así que BM25 acierta poco y la fusión le da la mitad del peso
igual. El re-ranking rescata la combinación —0,808, el mejor resultado— porque reordena leyendo cada
par pregunta-fragmento y descarta lo que BM25 metió por ruido.

La lectura correcta no es "el híbrido es mejor", que es lo que repiten los tutoriales, sino: **un
componente solo aporta si su supuesto se cumple**. Sobre un corpus en español, o con preguntas en
inglés, BM25 aportaría y la fusión ganaría. Aquí no, y solo se sabe midiendo. Si hubiera que llevar
esto a producción tal cual, lo defendible sería ponderar el RRF a favor del índice denso, o activar
BM25 únicamente cuando el idioma de la consulta coincide con el del corpus.

El Recall@1 de 0,385 contra 0,808 en Recall@10 dice además que el sistema **casi siempre encuentra el
artículo, pero no siempre lo pone primero**. Para este caso de uso está bien: los cinco fragmentos van
igual al prompt y el modelo elige. Sería un problema si se mostrara un único resultado al usuario.

### Limitaciones conocidas, declaradas a propósito

1. **No es un problema nuevo.** CUAD, ACORD y LegalBench-RAG existen para atacarlo. El ángulo
   diferencial de este trabajo es el chunking por estructura de cláusula sobre PDFs reales más la
   verificación automática de citas, no el retrieval en sí.
2. **RAG reduce la alucinación, no la elimina.** El validador detecta citas inventadas, pero no una
   interpretación equivocada de una cita correcta.
3. **Las referencias cruzadas rompen el supuesto del chunk autocontenido** ("sujeto a lo dispuesto en
   la Sección 12.3"). Cortar por artículo resuelve las referencias internas; las que apuntan a otro
   artículo siguen abiertas.
4. **La detección de estructura es heurística.** Funciona en los cuatro contratos del corpus, pero un
   PDF escaneado sin capa de texto, o uno sin numeración de artículos, cae al camino de respaldo y
   pierde calidad. No hay OCR en el pipeline.
5. **El corpus está en inglés** y bajo derecho estadounidense; las conclusiones no se transfieren
   directamente a contratos chilenos.
6. **El set de evaluación es pequeño** (26 preguntas, 4 contratos) y lo escribió quien construyó el
   sistema, que es la peor persona para escribirlo. Para cifras defendibles hay que correrlo sobre
   LegalBench-RAG completo.
7. **No es asesoría legal.** El sistema localiza y explica cláusulas; toda salida requiere revisión
   humana.
